# Level 4B — Graphical Analysis

**Audience:** analysts who want to explore conditional relationships and
clusters across a multi-asset return universe.

**Prerequisites:** Levels 1–2, basic covariance concepts, and the optional
`graphical` or `visualization` dependencies.

**Learning goals**

1. distinguish correlation from sparse partial correlation;
2. fit a labelled dependency network with deterministic clustering and layout;
3. inspect strong positive and negative conditional links;
4. create a Plotly network figure and a separate Dash application.

**Outline:** synthetic universe → fit → clusters → edges → Plotly → Dash
factory → threshold exercise.

The notebook uses synthetic monthly returns. It does not identify causal
relationships, recommend securities, or require credentials or network access.

## 1. Setup

The production calculation and visualization layers are deliberately separate:

- `graphical_analysis` estimates covariance, precision, partial correlation,
  clusters, edges, and two-dimensional coordinates;
- `visualization` converts the result into a Plotly figure;
- `dashboard` creates an optional Dash application around that figure.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.dashboard import (
    create_graphical_analysis_dashboard,
)
from asset_management_toolkit.graphical_analysis import graphical_analysis
from asset_management_toolkit.visualization import (
    dependency_network_figure,
)

## 2. Build a synthetic multi-asset universe

Two latent factors create related assets, while Gold and Cash add weaker
connections. Independent noise prevents exact duplicates. Labels remain attached
throughout the workflow.

In [ ]:
generator = np.random.default_rng(17)
n_observations = 120
growth_factor = generator.normal(0.006, 0.035, n_observations)
defensive_factor = generator.normal(0.003, 0.018, n_observations)
noise = generator.normal(0.0, 0.008, (n_observations, 8))

returns = pd.DataFrame(
    {
        "US Equity": growth_factor + noise[:, 0],
        "Developed Equity": 0.85 * growth_factor + noise[:, 1],
        "Emerging Equity": 1.10 * growth_factor + noise[:, 2],
        "Government Bond": defensive_factor + noise[:, 3],
        "Credit": 0.45 * growth_factor + 0.55 * defensive_factor + noise[:, 4],
        "Infrastructure": 0.55 * growth_factor + 0.25 * defensive_factor + noise[:, 5],
        "Gold": generator.normal(0.003, 0.025, n_observations) + noise[:, 6],
        "Cash": generator.normal(0.001, 0.002, n_observations) + noise[:, 7] * 0.10,
    },
    index=pd.date_range("2016-01-31", periods=n_observations, freq="ME"),
)
returns.describe().loc[["mean", "std"]].T

## 3. Fit the sparse dependency network

`GraphicalLassoCV` estimates a sparse precision matrix after standardization.
Off-diagonal precision terms are converted to signed partial correlations.
Affinity propagation assigns cluster labels, and metric MDS supplies stable
two-dimensional coordinates for display.

The edge threshold controls what is reported and plotted; it does not refit the
underlying covariance model.

In [ ]:
network = graphical_analysis(
    returns,
    edge_threshold=0.05,
    cv=5,
    random_state=7,
)

pd.DataFrame(
    {
        "cluster": network.cluster_labels,
        "x": network.embedding["x"],
        "y": network.embedding["y"],
    }
).sort_values(["cluster", "x"])

## 4. Inspect the strongest conditional links

Partial correlation asks whether two assets remain related after accounting
for the other assets in the fitted universe. A weak edge does not imply that
their ordinary pairwise correlation is zero.

In [ ]:
network.edges.head(12)

In [ ]:
comparison = pd.DataFrame(
    {
        "ordinary_correlation": returns.corr().stack(),
        "partial_correlation": network.partial_correlations.stack(),
    }
)
comparison = comparison[
    comparison.index.get_level_values(0)
    < comparison.index.get_level_values(1)
]
comparison.reindex(
    comparison["partial_correlation"].abs().sort_values(ascending=False).index
).head(12)

## 5. Create the Plotly research view

Edge width represents absolute partial-correlation strength. Teal edges are
positive and rose edges are negative. Node color represents cluster; hover text
reports the cluster and number of visible links.

In [ ]:
figure = dependency_network_figure(
    network,
    title="Synthetic multi-asset dependency network",
)
figure

## 6. Filter without refitting

Filtering changes only the displayed subgraph. The fitted precision matrix and
cluster assignments remain those of the complete universe.

In [ ]:
selected_cluster = [int(network.cluster_labels.loc["US Equity"])]
cluster_figure = dependency_network_figure(
    network,
    clusters=selected_cluster,
    title="Cluster containing US Equity",
)
cluster_figure

## 7. Create the Dash application

The factory returns a standard Dash app with a cluster selector and responsive
Plotly graph. A notebook should not start a long-running web server, so this
cell validates the app structure only.

To run it from a Python script:

```python
app = create_graphical_analysis_dashboard(network)
app.run(debug=False)
```

In [ ]:
app = create_graphical_analysis_dashboard(
    network,
    title="Synthetic Asset Dependency Network",
)
{
    "app_title": app.title,
    "layout_type": type(app.layout).__name__,
    "callback_count": len(app.callback_map),
}

## 8. Exercise — change the reporting threshold

Refit with `edge_threshold=0.15`, then compare:

1. the number of reported edges;
2. cluster labels and embedding coordinates;
3. the strongest retained positive and negative links.

Which outputs should remain unchanged, and why?

In [ ]:
# Try it here.
stricter = graphical_analysis(
    returns,
    edge_threshold=0.15,
    cv=5,
    random_state=7,
)

### Answer scaffold

In [ ]:
pd.Series(
    {
        "edges_at_0.05": len(network.edges),
        "edges_at_0.15": len(stricter.edges),
        "clusters_identical": network.cluster_labels.equals(
            stricter.cluster_labels
        ),
        "embedding_max_abs_difference": (
            network.embedding - stricter.embedding
        ).abs().to_numpy().max(),
    }
)

## Interpretation, pitfalls, and extensions

- Partial correlation is conditional association, not causality.
- Results depend on the selected universe, sample window, scaling, and
  regularization selected by cross-validation.
- Cluster numbers are arbitrary labels; interpret their members, not the
  numeric value.
- MDS coordinates support visualization and can rotate or reflect without
  changing pairwise geometry.
- Do not silently drop incomplete assets. Align and document the sample before
  fitting.
- A display threshold hides small edges but does not change the fitted model.
- Validate stability across chronological windows before using the network in
  a research decision.

Possible extensions include rolling networks, sector metadata, cluster
stability diagnostics, and downstream risk-budget comparisons. Each needs a
separate timing and validation contract.